# E09 Mall Customers：缩放、聚类数与稳定性

> Issue #9 / Epic #6 / 路线图 E09

## 案例定位
商场客户细分是**无监督聚类**经典题：没有「标准答案标签」，也没有 Kaggle 式提交排名。  
目标是：**用可重复的簇结构支撑运营动作**（高净值低消费 vs 低净值高消费等），而不是 chasing leaderboard。

## 你将学到
1. 聚类前必须做**特征缩放**（Income / Score 量纲不同）
2. **选 k**：肘部法（inertia）+ 轮廓系数（silhouette）互相印证
3. **稳定性**：换 `random_state` 时，标签置换不应剧烈变化（用 ARI 量化）
4. **wanglaoshi**：`DataAnalyzer` 做分布摸底；本案例**不做** `SubmissionGenerator`（非竞赛提交场景）

## 理论：K-Means 的假设与局限
- 假设：球形簇、近似等方差；对 Income-Score 散点通常尚可
- `n_init`：多次初始化取最优 inertia，减轻局部最优
- 稳定性：客户细分要可解释 → 若换种子簇大幅重排，业务难以落地

## 数据
- 本地：`data/Mall_Customers.csv`
- 缺失：合成客户画像 DEMO


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 1. 加载或合成


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/Mall_Customers.csv')
DEMO = False
if path.exists():
    df = pd.read_csv(path)
    print('local', df.shape)
else:
    DEMO = True
    rng = np.random.RandomState(RANDOM_STATE)
    n = 400 if QUICK else 800
    df = pd.DataFrame({
        'CustomerID': np.arange(1, n + 1),
        'Genre': rng.choice(['Male', 'Female'], n),
        'Age': rng.randint(18, 70, n),
        'Annual Income (k$)': rng.normal(60, 25, n).clip(15, 140),
        'Spending Score (1-100)': rng.normal(50, 25, n).clip(1, 100),
    })
    print('DEMO customers', df.shape)
df.head()

import hashlib
data_mode = 'demo' if DEMO else 'real'
n_rows = int(len(df))
if DEMO:
    data_source = 'synthetic_demo'
    data_sha256 = None
else:
    data_source = str(path.resolve()) if path.exists() else ('openml:' + str(path.name))
    try:
        if path.exists():
            data_sha256 = hashlib.sha256(path.read_bytes()).hexdigest()
        else:
            data_sha256 = hashlib.sha256(df.to_csv(index=False).encode('utf-8')).hexdigest()
            if not str(data_source).startswith('openml'):
                data_source = f'inmemory:{path.name}'
    except Exception:
        data_sha256 = None
print({'data_mode': data_mode, 'n_rows': n_rows, 'data_source': data_source, 'data_sha256': data_sha256})


## 2. EDA


In [ ]:
from wanglaoshi.analyze import DataAnalyzer

analyzer = DataAnalyzer(df)
display(analyzer.get_basic_info())
display(analyzer.basic_statistics())


## 3. 选数值特征与散点直觉


In [ ]:
import matplotlib.pyplot as plt

cols = []
for name in ['Annual Income (k$)', 'Spending Score (1-100)', 'Age']:
    if name in df.columns:
        cols.append(name)
if len(cols) < 2:
    cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cols = [c for c in cols if 'id' not in c.lower()][:3]

X = df[cols].astype(float)
print('cluster features:', cols)
if len(cols) >= 2:
    plt.figure(figsize=(5, 4))
    plt.scatter(X[cols[0]], X[cols[1]], alpha=0.5, s=20)
    plt.xlabel(cols[0])
    plt.ylabel(cols[1])
    plt.title('Income vs Spending（未缩放）')
    plt.tight_layout()
    plt.show()


## 4. 缩放 + 搜 k（肘部 + Silhouette）

轮廓系数 \(s \in [-1,1]\)：越接近 1 表示点与同簇更紧、与他簇更疏。  
肘部法看 inertia 下降放缓的 k；两者不一致时，优先结合**业务可解释性**定 k。


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

ks = range(2, 10 if not QUICK else 8)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lab = km.fit_predict(Xs)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Xs, lab))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(list(ks), inertias, marker='o')
ax[0].set_title('Elbow (inertia)')
ax[0].set_xlabel('k')
ax[1].plot(list(ks), sils, marker='o')
ax[1].set_title('Silhouette')
ax[1].set_xlabel('k')
plt.tight_layout()
plt.show()
best_k = list(ks)[int(np.argmax(sils))]
print('best_k by silhouette', best_k, 'score', round(max(sils), 4))


## 5. 多随机种子稳定性（ARI）

**Adjusted Rand Index (ARI)** 衡量两套标签在忽略置换意义下的一致性，1 为完全一致。  
对固定 `best_k`，扫描多个 `random_state`；若 ARI 均值过低，说明解不稳定，不宜直接用于运营命名。


In [ ]:
from sklearn.metrics import adjusted_rand_score

base = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10).fit_predict(Xs)
seeds = list(range(12 if QUICK else 30))
aris = []
for seed in seeds:
    lab = KMeans(n_clusters=best_k, random_state=seed, n_init=10).fit_predict(Xs)
    aris.append(adjusted_rand_score(base, lab))
print('ARI vs base seed: mean', round(np.mean(aris), 4), 'min', round(np.min(aris), 4))
print('sample ARI', aris[:8])


## 6. 最终簇标签与画像表


In [ ]:
labels = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10).fit_predict(Xs)
profile = df.copy()
profile['cluster'] = labels
centers = profile.groupby('cluster')[cols].mean()
display(centers)
if len(cols) >= 2:
    plt.figure(figsize=(5, 4))
    for c in range(best_k):
        m = labels == c
        plt.scatter(Xs[m, 0], Xs[m, 1], alpha=0.55, s=18, label=f'cluster {c}')
    plt.xlabel(cols[0] + ' (scaled)')
    plt.ylabel(cols[1] + ' (scaled)')
    plt.legend()
    plt.title(f'KMeans k={best_k}')
    plt.tight_layout()
    plt.show()


## 6.1 未缩放 vs 缩放：轮廓系数对照


In [ ]:
km_raw = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10).fit_predict(X.values)
km_scl = labels
print('silhouette raw', round(silhouette_score(X.values, km_raw), 4))
print('silhouette scaled', round(silhouette_score(Xs, km_scl), 4))


## 6.2 类别变量 `Genre` 与簇的列联


In [ ]:
if 'Genre' in df.columns:
    ct = pd.crosstab(profile['cluster'], profile['Genre'], normalize='index')
    display(ct)
else:
    print('无 Genre 列')


## 6.3 理论：ARI 与种子敏感性

ARI 接近 1 说明簇划分在标签置换意义下稳定。  
若 `min(ARI) < 0.7`，建议增大 `n_init`、尝试 GMM（软分配）或与业务专家共创 k。  
**本案例不是提交竞赛**：无需 `SubmissionGenerator`；交付物是「可解释的 `CustomerID → cluster` 映射表」。


## 6.4 拓展阅读

- **Davies–Bouldin**：簇内散 / 簇间距，越小越好（与 silhouette 互补）
- **层次聚类**：小 n 时可画树状图辅助选 k
- **SCatter 经典四象限**：Income × Spending 平面上的「土豪 / 潜力 / 谨慎 / 普通」叙事


## 7. 与竞赛流程的区别

| 维度 | 本案例 | 典型 Kaggle 分类 |
|------|--------|------------------|
| 标签 | 无真值簇标签 | 有监督 y |
| 指标 | 轮廓/ARI/业务解读 | AUC/LogLoss |
| 提交 | **不需要** | SubmissionGenerator |

若要将簇用于_campaign_，导出 `CustomerID, cluster` 即可，无需概率型预测文件。


## 8. 检查清单
- [x] 缩放后聚类
- [x] elbow + silhouette 选 k
- [x] 多 seed ARI 稳定性
- [x] 簇均值画像
- [ ] 业务命名（如「理性消费者」「冲动型」）
- [ ] 与 A/B 活动对照验证


## 附录：聚类实战 Q&A（中文）

**Q：k 越大越好吗？**  
A：否。k 增大 inertia 单调降，但 silhouette 常先升后降。要在可解释性与拟合间折中。

**Q：为何不用 Accuracy？**  
A：无标签场景根本没有 Accuracy。若用外部分群标签评估，那是另一套监督指标，不是本节重点。

**Q：K-Means 对离群点？**  
A：敏感。营销金额尖峰客户可单独做 anomaly 层，再对其子集聚类。

**Q：稳定性多重要？**  
A：运营要固定话术与 coupon 策略，ARI<0.8 的解难以年度复用。可多 seed 投票或选 medoid 代表点。

**Q：与 wanglaoshi 的关系？**  
A：`DataAnalyzer` 做分布与缺失；聚类本身用 sklearn。本案例 deliberately 不用 `SubmissionGenerator`，强调「洞察交付」而非排名。

**Q：Gender/年龄要进模型吗？**  
A：伦理与法规允许时可用于画像；若仅作切片报告，可不加入距离计算，避免敏感属性驱动分群。


## 附录 B：导出簇分配（非竞赛提交）


In [ ]:
from pathlib import Path
export = profile[['CustomerID', 'cluster'] + cols] if 'CustomerID' in profile.columns else profile[['cluster'] + cols]
out_path = Path('outputs/09_mall_customers')
out_path.mkdir(parents=True, exist_ok=True)
export.head(10).to_csv(out_path / 'cluster_assignments_preview.csv', index=False)
print('wrote', out_path / 'cluster_assignments_preview.csv', 'rows preview', len(export.head(10)))


## 附录 C：课堂讨论题

1. 若 Marketing 坚持 k=6 但 silhouette 峰值在 k=4，你如何沟通？  
2. ARI 很高但簇中心业务难命名，是否仍上线？  
3. 新增线上特征（会员等级）后，旧簇映射如何迁移？  
4. 对比 K-Means 与 Gaussian Mixture 的软分配优劣。


## 读本：客户细分项目里程碑（非竞赛）

| 阶段 | 产出 | 成功标准 |
|------|------|----------|
| W1 | 清洗 + EDA | 无重复 CustomerID，缺失率表 |
| W2 | 缩放 + 选 k | silhouette≥0.25 且业务认可 k |
| W3 | 稳定性 | ARI mean≥0.85 |
| W4 | 画像 + 命名 | 每簇一句话价值主张 |
| W5 | 试点 | 小规模 campaign 对照组 uplift |

聚类 ≠ 分类：没有唯一真值标签，因此**不要为了虚构 leaderboard 造 submission 文件**。wanglaoshi 的价值在 `DataAnalyzer` 帮你在 W1 快速发现量纲与缺失；后续距离与聚类仍用 sklearn 标准栈。

### 距离、尺度与特征权重

欧氏距离对量纲敏感：Income 以千美元计、Score 0–100，若不 `StandardScaler`，Income 将主导距离。  
若业务认为 Spending 更重要，可在缩放后对列乘权重（例如 `Xs[:,1] *= 1.5`），但须在文档写明，否则年度复现困难。

### 从簇到行动

常见四象限叙事（Income × Spending）：高收高花「喜爱型」、高收低花「谨慎型」、低收高花「冲动型」、低收低花「普通型」。数字标签需经 centroid 验证，不可生搬硬套。


## 教师导读：mall customers 课堂 90 分钟脚本

**0–10 min**：展示未缩放散点，问「哪个轴主导距离？」引导学生说 Income。  
**10–25 min**：`StandardScaler` + 重新散点；强调聚类是几何问题。  
**25–40 min**：肘部 + silhouette 双图；出现分歧时投票，不要求唯一 k。  
**40–55 min**：多 seed ARI；若 min ARI 低，现场调高 n_init 示范。  
**55–70 min**：簇画像 + 四象限叙事；讨论「簇不等于人格标签」的伦理。  
**70–85 min**：对比 Kaggle 分类赛：无 submission、无 PR-AUC。  
**85–90 min**：收口 wanglaoshi：`DataAnalyzer` 在 W1，聚类在 sklearn。

### 学生常问

- 「silhouette 为负是否不能用？」部分点本就处于簇边界，可接受。  
- 「能否用 DBSCAN？」可以，但要讲密度参数 ε；本案例用 KMeans 因可重复。  
- 「Genre 要 one-hot 吗？」可选侧车分析，不必进距离，除非业务明确要按性别分群。

### 与仓库其它案例链接

- E05：监督 + 极端不均衡 + PR-AUC + 提交  
- E10：监督 + 高基数 TE 泄漏  
- E11：回归 + log 目标 + ModelEnsemble  

本案例刻意**不**走提交流程，避免学生误认为所有数据集都要 `SubmissionGenerator`。


## 附录 B：稳定性实验设计

固定 k，改 `random_state` 多次，计算标签与参考运行的 ARI。不稳定 → 特征不足或 k 不合理。


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler
import numpy as np
try:
    Xn = df.select_dtypes('number')
    Xs = StandardScaler().fit_transform(Xn)
    ref = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(Xs)
    aris=[]
    for s in range(5 if QUICK else 15):
        lab=KMeans(n_clusters=5, n_init=10, random_state=s).fit_predict(Xs)
        aris.append(adjusted_rand_score(ref, lab))
    print('ARI mean±std vs seed0:', float(np.mean(aris)), float(np.std(aris)))
except Exception as e:
    print(e)


## 深度阅读与作业（Mall Customers）

### 必读观点
1. 无监督任务用轮廓系数与 **多种子稳定性**，不是 Accuracy。
2. k 的选择要能讲出业务分层（高消费/价格敏感等）。
3. 标准化影响距离型聚类结果。

### 书面作业
1. 报告 k=3..8 的轮廓系数曲线。
2. 计算多种子 ARI 稳定性。
3. 为最终簇写 3 句运营话术（禁止虚构因果）。


## 课堂讨论记录区（请学生填写）

主题：09_mall_customers

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。
